# Minggu 3 — Baseline Model: Decision Tree

In [1]:
import json
import time
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train_model.csv")
val_df = pd.read_csv(DATA_DIR / "val_model.csv")

with open(DATA_DIR / "feature_lists.json", "r", encoding="utf-8") as f:
    feature_lists = json.load(f)

with open(DATA_DIR / "class_weights.json", "r", encoding="utf-8") as f:
    class_weights_info = json.load(f)

features_full = feature_lists["model_columns"]
hs_cols = ["hs_count", "hs_count_7d", "hs_count_30d", "hs_days_since_last"]
features_weather = [c for c in features_full if c not in hs_cols]
target_col = feature_lists["target"]

cw_dict = {0: class_weights_info["0"], 1: class_weights_info["1"]}

X_train = train_df[features_full]
y_train = train_df[target_col].astype(int).values
X_val = val_df[features_full]
y_val = val_df[target_col].astype(int).values

print("Train shape:", X_train.shape, "| Validation shape:", X_val.shape)
print("Class weights diterapkan:", cw_dict)

start_dt = time.time()
dt_baseline = DecisionTreeClassifier(class_weight=cw_dict, random_state=42)
dt_baseline.fit(X_train, y_train)
durasi_dt = time.time() - start_dt

y_val_pred = dt_baseline.predict(X_val)
y_val_proba = dt_baseline.predict_proba(X_val)[:, 1]

start_dt_w = time.time()
dt_weather = DecisionTreeClassifier(class_weight=cw_dict, random_state=42)
dt_weather.fit(train_df[features_weather], y_train)
durasi_dt_w = time.time() - start_dt_w

y_val_pred_w = dt_weather.predict(val_df[features_weather])
y_val_proba_w = dt_weather.predict_proba(val_df[features_weather])[:, 1]

summary_dt = pd.DataFrame(
    [
        {
            "Skenario": "Decision Tree (Full - 28 Fitur)",
            "Durasi (dtk)": round(durasi_dt, 3),
            "Accuracy": round(accuracy_score(y_val, y_val_pred), 4),
            "Precision": round(precision_score(y_val, y_val_pred, zero_division=0), 4),
            "Recall": round(recall_score(y_val, y_val_pred, zero_division=0), 4),
            "F1-Score": round(f1_score(y_val, y_val_pred, zero_division=0), 4),
            "PR-AUC": round(average_precision_score(y_val, y_val_proba), 4),
            "ROC-AUC": round(roc_auc_score(y_val, y_val_proba), 4),
        },
        {
            "Skenario": "Decision Tree (Ablation Cuaca - 24 Fitur)",
            "Durasi (dtk)": round(durasi_dt_w, 3),
            "Accuracy": round(accuracy_score(y_val, y_val_pred_w), 4),
            "Precision": round(precision_score(y_val, y_val_pred_w, zero_division=0), 4),
            "Recall": round(recall_score(y_val, y_val_pred_w, zero_division=0), 4),
            "F1-Score": round(f1_score(y_val, y_val_pred_w, zero_division=0), 4),
            "PR-AUC": round(average_precision_score(y_val, y_val_proba_w), 4),
            "ROC-AUC": round(roc_auc_score(y_val, y_val_proba_w), 4),
        },
    ]
)

print("=== HASIL VALIDASI: BASELINE MODEL (DECISION TREE) ===")
print(summary_dt.to_string(index=False))
print("--- Classification Report (Full 28 Fitur) ---")
print(classification_report(y_val, y_val_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

joblib.dump(dt_baseline, MODELS_DIR / "baseline_decision_tree.pkl")
print("Model disimpan ke:", MODELS_DIR / "baseline_decision_tree.pkl")


Train shape: (8586, 28) | Validation shape: (2148, 28)
Class weights diterapkan: {0: 0.5555124223602484, 1: 5.003496503496503}


=== HASIL VALIDASI: BASELINE MODEL (DECISION TREE) ===
                                 Skenario  Durasi (dtk)  Accuracy  Precision  Recall  F1-Score  PR-AUC  ROC-AUC
          Decision Tree (Full - 28 Fitur)         0.257    0.8599     0.6757  0.6194    0.6463  0.4972   0.7710
Decision Tree (Ablation Cuaca - 24 Fitur)         0.150    0.8198     0.5707  0.5180    0.5431  0.3953   0.7082
--- Classification Report (Full 28 Fitur) ---
              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1704
           1       0.68      0.62      0.65       444

    accuracy                           0.86      2148
   macro avg       0.79      0.77      0.78      2148
weighted avg       0.86      0.86      0.86      2148

Confusion Matrix:
[[1572  132]
 [ 169  275]]
Model disimpan ke: D:\MATKUL\SEMESTER 5\Cloud Computing\UAS\Prediksi-Kobong\models\baseline_decision_tree.pkl
